# 04. Tree Speculative Decoding: Medusa, EAGLE, Graph-of-Thoughts, and Jacobi Lookahead

- **Category:** 15_compression_peft_and_speculative_inference / 02_adaptive_kv_speculative_and_merging
- **Lineage & Citations:** Cai et al. (2024, *Medusa: Simple LLM Inference Acceleration with Multiple Decoding Heads*); Li et al. (ICML 2024, *EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty*); Besta et al. (AAAI 2024, *Graph of Thoughts: Solving Elaborate Problems with Large Language Models*); Fu et al. (2024, *Break the Sequential Dependency of LLM Inference with Jacobi Decoding*).
- **Purpose:** Implement multi-head speculative drafting (Medusa), construct 2D causal tree attention masks, extract longest accepted tree paths, and execute parallel Jacobi lookahead iterations.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Speculative Decoding Lossless Rejection Sampling and Drafting](03_speculative_decoding_lossless_rejection_sampling_and_drafting.ipynb) $\to$ `🔗 Downstream Syntheses:` [Model Merging Slerp Task Arithmetic TIES DARE and Evolutionary](05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never use standard lower-triangular causal attention masks on flattened tree speculative nodes; cross-branch attention will leak alternative speculative realities into sibling hidden states, corrupting verification. Avoid wide trees ($N > 64$ nodes) on memory-constrained GPUs where attention matrix expansion slows down the verification step.

## Mathematical Foundations & Governing Equations

### 1. Medusa Multi-Head Drafting
$$z_t^{(k)} = h_t W^{(k)} + b^{(k)}, \quad \tilde{x}_{t+k} = \arg\max_v z_t^{(k)}[v], \quad k \in \{1, \dots, K\}$$

### 2. 2D Causal Tree Attention Mask
For query node $i$ and key node $j$ in flattened candidate tree:
$$M_{ij} = \begin{cases} 0.0 & \text{if } j \in \text{Ancestors}(i) \lor j = i \\ -10^9 & \text{otherwise} \end{cases}$$

### 3. Longest Continuous Accepted Tree Path
$$\text{Accepted}(i) = \text{Accepted}(\text{Parent}(i)) \land (\tilde{x}_i == x_i^{\text{target}}), \quad L^* = \max_{i \in \text{Leaves}} \text{Depth}(i)$$

### 4. Jacobi Non-Autoregressive Fixed-Point Iteration
$$X^{(k+1)} = \tanh(X^{(k)} W + B), \quad \text{Residual} = \|X^{(k+1)} - X^{(k)}\|_\infty$$

### Complexity
Evaluates an entire candidate tree of depth $K$ and branching factor $B$ in a single $\mathcal{O}(1)$ target forward pass, boosting throughput by $2.5\times - 4\times$.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def medusa_heads_predict(hidden, head_weights, head_biases):
    num_heads = head_weights.shape[0]
    logits_list, top_tokens_list = [], []
    for k in range(num_heads):
        hl = hidden @ head_weights[k] + head_biases[k]
        logits_list.append(hl)
        top_tokens_list.append(torch.argmax(hl, dim=-1))
    return torch.stack(logits_list, dim=1), torch.stack(top_tokens_list, dim=1)

def build_tree_attention_mask(parent_indices):
    n = parent_indices.shape[0]
    mask = torch.full((n, n), -1e9, dtype=torch.float32, device=parent_indices.device)
    for i in range(n):
        curr = i
        visited = set()
        while curr not in visited:
            mask[i, curr] = 0.0
            visited.add(curr)
            if curr == 0:
                break
            curr = int(parent_indices[curr].item())
    return mask

def tree_longest_accepted_path(candidate_tokens, target_tokens, parent_indices):
    n = candidate_tokens.shape[0]
    matches = (candidate_tokens == target_tokens)
    accepted = [False] * n
    accepted[0] = True
    for i in range(1, n):
        p = int(parent_indices[i].item())
        if accepted[p] and matches[i].item():
            accepted[i] = True
    depths = [0] * n
    best_len, best_node = 0, 0
    for i in range(n):
        if accepted[i]:
            depths[i] = 1 if i == 0 else depths[int(parent_indices[i].item())] + 1
            if depths[i] > best_len:
                best_len = depths[i]
                best_node = i
    path_mask = torch.zeros(n, dtype=torch.float32, device=candidate_tokens.device)
    curr = best_node
    while True:
        path_mask[curr] = 1.0
        if curr == 0:
            break
        curr = int(parent_indices[curr].item())
    return path_mask, torch.tensor(float(best_len), dtype=torch.float32, device=candidate_tokens.device)

def jacobi_lookahead_step(x, weight, bias):
    x_next = torch.tanh(x @ weight + bias)
    res, _ = torch.max(torch.abs(x_next - x).flatten(start_dim=1), dim=-1)
    return x_next, res

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

m_logits, t_tokens = medusa_heads_predict(weights["hidden"], weights["head_weights"], weights["head_biases"])
assert float(torch.max(torch.abs(m_logits - weights["medusa_logits"]))) < 1e-5
assert float(torch.max(torch.abs(t_tokens.to(torch.float32) - weights["top_tokens"]))) < 1e-5

t_mask = build_tree_attention_mask(weights["parent_indices"].to(torch.int64))
assert float(torch.max(torch.abs(t_mask - weights["tree_mask"]))) < 1e-5

p_mask, p_len = tree_longest_accepted_path(weights["candidate_tokens"].to(torch.int64), weights["target_tokens"].to(torch.int64), weights["parent_indices"].to(torch.int64))
assert float(torch.max(torch.abs(p_mask - weights["path_mask"]))) < 1e-5
assert float(torch.max(torch.abs(p_len - weights["path_length"]))) < 1e-5

x_nxt, j_res = jacobi_lookahead_step(weights["x_jacobi"], weights["w_jacobi"], weights["b_jacobi"])
assert float(torch.max(torch.abs(x_nxt - weights["x_next"]))) < 1e-5
assert float(torch.max(torch.abs(j_res - weights["jacobi_res"]))) < 1e-5
print("All 5 tree speculative decoding modules match golden checkpoints (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_medusa = benchmark_fn(lambda: medusa_heads_predict(weights["hidden"], weights["head_weights"], weights["head_biases"]), iters=100, name="Medusa 3-Head Predict")
b_tree = benchmark_fn(lambda: build_tree_attention_mask(weights["parent_indices"].to(torch.int64)), iters=100, name="Tree Mask Gen (6-node)")
b_jacobi = benchmark_fn(lambda: jacobi_lookahead_step(weights["x_jacobi"], weights["w_jacobi"], weights["b_jacobi"]), iters=100, name="Jacobi Lookahead Step")
print(format_benchmark_table([b_medusa, b_tree, b_jacobi]))

print(f"Accepted Tree Path: {int(p_len.item())} tokens accepted in single target verification pass!")

In [ ]:
# Pathological Misuse: Naive lower-triangular causal mask permits cross-branch contamination
parents = torch.tensor([0, 0, 0, 1, 1, 2])
tree_mask = build_tree_attention_mask(parents)

# Node 4 is child of node 1; Node 5 is child of node 2 (sibling branches!)
# Under 2D tree mask: Node 5 CANNOT attend to Node 4 (value is -1e9)
assert tree_mask[5, 4].item() == -1e9, "Tree mask must forbid cross-branch attention between sibling nodes!"

# Under naive lower-triangular causal mask: position 5 can see position 4 because 4 < 5!
naive_mask = torch.triu(torch.full((6, 6), -1e9), diagonal=1)
assert naive_mask[5, 4].item() == 0.0, "Naive causal mask incorrectly allows sibling branch leakage!"
print("Verified: Naive causal masks cause sibling branch leakage; custom 2D tree mask is mathematically required.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Tree Exclusivity**: Verifying non-linear candidate branches requires a 2D tree attention mask to prevent sibling context leakage.
2. **Zero Secondary Model Overhead**: Medusa heads leverage the frozen base model's internal representations, eliminating the memory and operational burden of a separate draft LLM.
3. **Relaxation Over Recurrence**: Jacobi fixed-point iteration updates future tokens in parallel, bypassing autoregressive serialization for structured text.

---
### Cross-Domain Generalisation Matrix
- **Decision Trees & Monte Carlo Tree Search (MCTS):** Tree speculative verification evaluates multiple exploration rollouts in parallel.
- **Distributed Ray Tracing:** Primary and secondary reflection rays traverse spatial hierarchy trees with ancestry visibility.
- **Numerical PDE Solvers:** Jacobi relaxation in parallel linear algebra solves grid updates synchronously across spatial cells.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_tree_speculative_medusa_eagle_got_and_jacobi.md](../../../markdown/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.md)
- **PyTorch 2.x Module:** [04_tree_speculative_medusa_eagle_got_and_jacobi.py](../../../pytorch2/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.py)
- **Functional JAX Module:** [04_tree_speculative_medusa_eagle_got_and_jacobi.py](../../../jax/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.py)
- **Native Rust Candle Track:** [04_tree_speculative_medusa_eagle_got_and_jacobi.rs](../../../candle/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.rs)
- **Hardware Kernel (CPU / WGSL):** [04_tree_speculative_medusa_eagle_got_and_jacobi.rs](../../../kernels/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.rs)
- **Golden SafeTensors:** [04_tree_speculative_medusa_eagle_got_and_jacobi.safetensors](../../../weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/04_tree_speculative_medusa_eagle_got_and_jacobi.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Speculative Decoding Lossless Rejection Sampling and Drafting](03_speculative_decoding_lossless_rejection_sampling_and_drafting.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 05. Model Merging Slerp Task Arithmetic TIES DARE and Evolutionary](05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.ipynb) &rarr;